# ItalyNERD Reproducible Pipeline

Notebook unico e riproducibile per: dataset study italiano, valutazione NER, error analysis ed export artefatti per il paper.

In [ ]:
# 1) Setup ambiente e seed
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer
)
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path.cwd()
ART = ROOT / 'benchmark_results' / 'italy_notebook_artifacts'
IMG = ROOT / 'images'
ART.mkdir(parents=True, exist_ok=True)
IMG.mkdir(parents=True, exist_ok=True)

print(f'Working dir: {ROOT}')
print(f'Artifacts dir: {ART}')

In [ ]:
# 1b) Environment capture per allineamento paper (paper-locked)
import sys
import platform
from datetime import datetime, timezone
import importlib.metadata as md

def _pkg_ver(name: str):
    try:
        return md.version(name)
    except Exception:
        return "not-installed"

ENV_INFO = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": _pkg_ver("torch"),
    "transformers": _pkg_ver("transformers"),
    "datasets": _pkg_ver("datasets"),
    "seqeval": _pkg_ver("seqeval"),
    "evaluate": _pkg_ver("evaluate"),
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_device_count": int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

with open(ART / "paper_claims_environment.json", "w", encoding="utf-8") as f:
    json.dump(ENV_INFO, f, indent=2)

print("Saved environment:", ART / "paper_claims_environment.json")
print(ENV_INFO)

In [ ]:
# 2) Configurazione run
MODEL_NAME = 'roberta-base'
HF_DATASET_NAME = 'Babelscape/multinerd'
HF_REVISION = 'refs/convert/parquet'
LANGS = ['it']  # default: solo italiano
MAX_LENGTH = 512

# Se True, il notebook FALLISCE se il checkpoint locale non esiste (evita fallback non voluto)
REQUIRE_LOCAL_MODEL_FOR_PAPER = True

# Se hai già un modello fine-tunato, imposta qui il path (es: multinerd_checkpoints/final_model_it)
LOCAL_FINETUNED_MODEL = ROOT / 'multinerd_checkpoints' / 'final_model_it'

print('LANGS:', LANGS)
print('Local model exists:', LOCAL_FINETUNED_MODEL.exists())
print('REQUIRE_LOCAL_MODEL_FOR_PAPER:', REQUIRE_LOCAL_MODEL_FOR_PAPER)

In [ ]:
# 3) Label map canonico MultiNERD (31 labels)
LABEL_LIST = [
    'O',
    'B-PER', 'I-PER',
    'B-ORG', 'I-ORG',
    'B-LOC', 'I-LOC',
    'B-ANIM', 'I-ANIM',
    'B-BIO', 'I-BIO',
    'B-CEL', 'I-CEL',
    'B-DIS', 'I-DIS',
    'B-EVE', 'I-EVE',
    'B-FOOD', 'I-FOOD',
    'B-INST', 'I-INST',
    'B-MEDIA', 'I-MEDIA',
    'B-MYTH', 'I-MYTH',
    'B-PLANT', 'I-PLANT',
    'B-TIME', 'I-TIME',
    'B-VEHI', 'I-VEHI'
]
ID2LABEL = {i: l for i, l in enumerate(LABEL_LIST)}
LABEL2ID = {l: i for i, l in ID2LABEL.items()}

assert len(LABEL_LIST) == 31
print('Num labels:', len(LABEL_LIST))

In [ ]:
# 4) Caricamento dataset e filtro lingua
raw = load_dataset(HF_DATASET_NAME, revision=HF_REVISION)

def keep_lang(example):
    return example['lang'] in LANGS

dataset = DatasetDict({
    split: raw[split].filter(keep_lang)
    for split in ['train', 'validation', 'test']
})

split_sizes = {k: len(v) for k, v in dataset.items()}
print('Split sizes:', split_sizes)
pd.DataFrame([split_sizes]).to_csv(ART / 'italian_split_sizes.csv', index=False)

In [ ]:
# 5) Visualizzazione split sizes (ItalianSplitOverview.png)
df = pd.DataFrame({'split': list(split_sizes.keys()), 'count': list(split_sizes.values())})

plt.figure(figsize=(7,4))
bars = plt.bar(df['split'], df['count'])
plt.title('Italian-only split sizes')
plt.xlabel('Split')
plt.ylabel('Examples')
for b in bars:
    h = b.get_height()
    plt.text(b.get_x()+b.get_width()/2, h, f'{int(h):,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
out_path = IMG / 'ItalianSplitOverview.png'
plt.savefig(out_path, dpi=200)
plt.show()
print('Saved:', out_path)

In [ ]:
# 6) Dataset study: densità token/entity + distribuzione entity types
def count_entities_b(ner_tags):
    c = 0
    for t in ner_tags:
        if t < len(LABEL_LIST) and LABEL_LIST[t].startswith('B-'):
            c += 1
    return c

train_df = pd.DataFrame({
    'token_count': [len(x['tokens']) for x in dataset['train']],
    'entity_count': [count_entities_b(x['ner_tags']) for x in dataset['train']]
})

train_df.to_csv(ART / 'italian_entity_token_density.csv', index=False)

# entity type distribution (B-tags only)
type_counts = {}
for ex in dataset['train']:
    for t in ex['ner_tags']:
        if t < len(LABEL_LIST):
            name = LABEL_LIST[t]
            if name.startswith('B-'):
                et = name[2:]
                type_counts[et] = type_counts.get(et, 0) + 1

type_df = pd.DataFrame({'entity_type': list(type_counts.keys()), 'count': list(type_counts.values())})
type_df = type_df.sort_values('count', ascending=False).reset_index(drop=True)
type_df.to_csv(ART / 'italian_entity_type_distribution.csv', index=False)

print('Density rows:', len(train_df))
print('Entity types:', len(type_df))

In [ ]:
# 7) Figure: EntityCountTokenCountIt_Recomputed.png
plt.figure(figsize=(6,4))
plt.scatter(train_df['token_count'], train_df['entity_count'], alpha=0.2, s=8)
plt.title('Entity count vs token count (Italian train)')
plt.xlabel('Token count')
plt.ylabel('Entity count')
plt.tight_layout()
out_path = IMG / 'EntityCountTokenCountIt_Recomputed.png'
plt.savefig(out_path, dpi=200)
plt.show()
print('Saved:', out_path)

In [ ]:
# 8) Figure: TokenCountDistributionIt.png
plt.figure(figsize=(6,4))
plt.hist(train_df['token_count'], bins=60, alpha=0.85)
plt.axvline(MAX_LENGTH, color='red', linestyle='--', label=f'max_length={MAX_LENGTH}')
plt.title('Token count distribution (Italian train)')
plt.xlabel('Tokens per sample')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
out_path = IMG / 'TokenCountDistributionIt.png'
plt.savefig(out_path, dpi=200)
plt.show()
print('Saved:', out_path)

In [ ]:
# 9) Figure: EntityDistributionAnalysis_It_Recomputed.png
top_n = 15
plot_df = type_df.head(top_n).iloc[::-1]
plt.figure(figsize=(7,5))
plt.barh(plot_df['entity_type'], plot_df['count'])
plt.title('Entity-type distribution (Italian train, B-tags)')
plt.xlabel('Count')
plt.tight_layout()
out_path = IMG / 'EntityDistributionAnalysis_It_Recomputed.png'
plt.savefig(out_path, dpi=200)
plt.show()
print('Saved:', out_path)

In [ ]:
# 10) Tokenizer + allineamento label
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH
    )

    aligned_labels = []
    for i, word_labels in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_wid = None
        label_ids = []

        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev_wid:
                label_ids.append(word_labels[wid] if wid < len(word_labels) else -100)
            else:
                if wid < len(word_labels):
                    current = word_labels[wid]
                    if current < len(LABEL_LIST) and LABEL_LIST[current].startswith('B-'):
                        i_name = 'I-' + LABEL_LIST[current][2:]
                        label_ids.append(LABEL2ID.get(i_name, -100))
                    else:
                        label_ids.append(current)
                else:
                    label_ids.append(-100)
            prev_wid = wid

        aligned_labels.append(label_ids)

    tokenized['labels'] = aligned_labels
    return tokenized

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
print(tokenized_datasets)

In [ ]:
# 11) Carica modello per valutazione (paper-locked)
if REQUIRE_LOCAL_MODEL_FOR_PAPER and (not LOCAL_FINETUNED_MODEL.exists()):
    raise FileNotFoundError(
        f"Checkpoint locale richiesto ma assente: {LOCAL_FINETUNED_MODEL}. "
        "Disattiva REQUIRE_LOCAL_MODEL_FOR_PAPER solo per test esplorativi."
    )

if LOCAL_FINETUNED_MODEL.exists():
    model_source = str(LOCAL_FINETUNED_MODEL)
else:
    model_source = MODEL_NAME

model = AutoModelForTokenClassification.from_pretrained(
    model_source,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

data_collator = DataCollatorForTokenClassification(tokenizer)
trainer = Trainer(model=model, tokenizer=tokenizer, data_collator=data_collator)
print('Model source:', model_source)

In [ ]:
# 12) Predizioni + metriche complete (+ runtime metrics)
pred_output = trainer.predict(tokenized_datasets['test'])
predictions = pred_output.predictions
labels = pred_output.label_ids
runtime_metrics = dict(pred_output.metrics) if hasattr(pred_output, 'metrics') else {}

pred_ids = np.argmax(predictions, axis=2)

true_labels = []
true_preds = []
for pred, gold in zip(pred_ids, labels):
    seq_gold = []
    seq_pred = []
    for p, g in zip(pred, gold):
        if g != -100:
            if g < len(LABEL_LIST) and p < len(LABEL_LIST):
                seq_gold.append(LABEL_LIST[g])
                seq_pred.append(LABEL_LIST[p])
    true_labels.append(seq_gold)
    true_preds.append(seq_pred)

metrics = {
    'precision': float(precision_score(true_labels, true_preds)),
    'recall': float(recall_score(true_labels, true_preds)),
    'f1': float(f1_score(true_labels, true_preds)),
    'accuracy': float(accuracy_score(true_labels, true_preds))
}

report_text = classification_report(true_labels, true_preds, digits=4)
print('Metrics:', metrics)
print('Runtime metrics:', runtime_metrics)
print(report_text[:1500])

In [ ]:
# 13) Export artefatti paper-ready + single source of truth (paper_claims.json)
with open(ART / 'test_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

with open(ART / 'classification_report.txt', 'w', encoding='utf-8') as f:
    f.write(report_text)

summary_df = pd.DataFrame([{
    'eval_precision': metrics['precision'],
    'eval_recall': metrics['recall'],
    'eval_f1': metrics['f1'],
    'eval_accuracy': metrics['accuracy']
}])
summary_df.to_csv(ART / 'test_metrics.csv', index=False)

env_path = ART / 'paper_claims_environment.json'
env_payload = {}
if env_path.exists():
    with open(env_path, 'r', encoding='utf-8') as f:
        env_payload = json.load(f)

paper_claims = {
    'run_scope': 'italian_primary',
    'dataset': {
        'name': HF_DATASET_NAME,
        'revision': HF_REVISION,
        'langs': LANGS,
        'split_sizes': split_sizes,
    },
    'model': {
        'model_source': model_source,
        'require_local_model_for_paper': REQUIRE_LOCAL_MODEL_FOR_PAPER,
        'max_length': MAX_LENGTH,
        'num_labels': len(LABEL_LIST),
    },
    'metrics_seqeval': metrics,
    'metrics_runtime': runtime_metrics,
    'environment': env_payload,
    'artifacts': {
        'test_metrics_json': str(ART / 'test_metrics.json'),
        'test_metrics_csv': str(ART / 'test_metrics.csv'),
        'classification_report_txt': str(ART / 'classification_report.txt'),
    }
}

with open(ART / 'paper_claims.json', 'w', encoding='utf-8') as f:
    json.dump(paper_claims, f, indent=2)

print('Saved:')
print('-', ART / 'test_metrics.json')
print('-', ART / 'test_metrics.csv')
print('-', ART / 'classification_report.txt')
print('-', ART / 'paper_claims.json')

In [ ]:
# 14) Error analysis (top transition errors)
from collections import Counter

errors = Counter()
for gold_seq, pred_seq in zip(true_labels, true_preds):
    for g, p in zip(gold_seq, pred_seq):
        if g != p:
            errors[(g, p)] += 1

top_errors = errors.most_common(30)
err_df = pd.DataFrame([{'true': g, 'pred': p, 'count': c} for (g,p), c in top_errors])
err_df.to_csv(ART / 'top_error_transitions.csv', index=False)

print('Top errors:')
display(err_df.head(15))

In [ ]:
# 15) Esempio output da log reale (ItalianExecutionOutputExample.png)
esito_path = ROOT / 'esito.txt'
if esito_path.exists():
    text = esito_path.read_text(encoding='utf-8', errors='ignore')
    marker = 'WikiNER (IT):'
    if marker in text:
        chunk = text.split(marker, 1)[1][:3500]
        render_text = marker + '\n' + chunk

        plt.figure(figsize=(14, 8))
        plt.axis('off')
        plt.text(0.01, 0.99, render_text, va='top', ha='left', family='monospace', fontsize=8)
        plt.tight_layout()
        out_path = IMG / 'ItalianExecutionOutputExample.png'
        plt.savefig(out_path, dpi=220, bbox_inches='tight')
        plt.show()
        print('Saved:', out_path)
    else:
        print('Marker WikiNER (IT) non trovato in esito.txt')
else:
    print('esito.txt non trovato, salto questa cella.')

## Note finali
- Tutti gli artefatti esportati sono in `benchmark_results/italy_notebook_artifacts/`.
- Le immagini usate nel paper sono salvate in `images/` con naming coerente.
- Se vuoi training completo nello stesso notebook, puoi aggiungere una sezione `TrainingArguments + Trainer.train()` prima della valutazione.

## Sezione A — Training Italiano vs Multilingua (separazione esplicita)

Questa sezione aggiunge pipeline separate:

- **A1: Training Italiano (`lang=it`)**
- **A2: Training Multilingua (`lang in ['it','en']`)**

Entrambe usano la stessa logica di tokenizzazione/allineamento e salvano artefatti in cartelle dedicate per evitare confusione.

In [ ]:
# A0) Utility comuni per training (riutilizzate da IT e IT+EN)
from dataclasses import dataclass
from typing import List, Dict
from transformers import TrainingArguments

@dataclass
class RunConfig:
    run_name: str
    langs: List[str]
    output_subdir: str
    learning_rate: float = 2e-5
    batch_size: int = 4
    grad_accum: int = 4
    epochs: int = 3
    weight_decay: float = 0.01
    warmup_steps: int = 5000


def load_filtered_multinerd(langs: List[str]):
    raw_local = load_dataset(HF_DATASET_NAME, revision=HF_REVISION)

    def keep(ex):
        return ex['lang'] in langs

    filtered = DatasetDict({
        split: raw_local[split].filter(keep)
        for split in ['train', 'validation', 'test']
    })
    return filtered


def tokenize_dataset(input_ds):
    return input_ds.map(tokenize_and_align_labels, batched=True)


def build_training_args(cfg: RunConfig):
    out_dir = ROOT / 'multinerd_checkpoints' / cfg.output_subdir
    out_dir.mkdir(parents=True, exist_ok=True)

    return TrainingArguments(
        output_dir=str(out_dir),
        eval_strategy='steps',
        eval_steps=5000,
        save_strategy='steps',
        save_steps=5000,
        learning_rate=cfg.learning_rate,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accum,
        num_train_epochs=cfg.epochs,
        weight_decay=cfg.weight_decay,
        warmup_steps=cfg.warmup_steps,
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        report_to='none',
        fp16=torch.cuda.is_available(),
        dataloader_pin_memory=False,
        dataloader_num_workers=0,
        save_total_limit=3,
        max_grad_norm=1.0,
        seed=SEED
    )


def compute_metrics_for_trainer(eval_pred):
    preds, gold = eval_pred
    preds = np.argmax(preds, axis=2)

    y_true, y_pred = [], []
    for p_seq, g_seq in zip(preds, gold):
        t_seq, pr_seq = [], []
        for p, g in zip(p_seq, g_seq):
            if g != -100 and g < len(LABEL_LIST) and p < len(LABEL_LIST):
                t_seq.append(LABEL_LIST[g])
                pr_seq.append(LABEL_LIST[p])
        y_true.append(t_seq)
        y_pred.append(pr_seq)

    return {
        'precision': float(precision_score(y_true, y_pred)),
        'recall': float(recall_score(y_true, y_pred)),
        'f1': float(f1_score(y_true, y_pred)),
        'accuracy': float(accuracy_score(y_true, y_pred))
    }


def train_and_evaluate(cfg: RunConfig):
    ds = load_filtered_multinerd(cfg.langs)
    tok_ds = tokenize_dataset(ds)

    local_model_dir = ROOT / 'multinerd_checkpoints' / cfg.output_subdir / 'final_model'
    model_name_or_path = str(local_model_dir) if local_model_dir.exists() else MODEL_NAME

    model_local = AutoModelForTokenClassification.from_pretrained(
        model_name_or_path,
        num_labels=len(LABEL_LIST),
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )

    args_local = build_training_args(cfg)
    trainer_local = Trainer(
        model=model_local,
        args=args_local,
        train_dataset=tok_ds['train'],
        eval_dataset=tok_ds['validation'],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics_for_trainer
    )

    train_output = trainer_local.train()
    eval_metrics = trainer_local.evaluate(tok_ds['test'])

    final_dir = ROOT / 'multinerd_checkpoints' / cfg.output_subdir / 'final_model'
    final_dir.mkdir(parents=True, exist_ok=True)
    trainer_local.save_model(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))

    pred_out = trainer_local.predict(tok_ds['test'])
    pred_ids = np.argmax(pred_out.predictions, axis=2)
    labels_ids = pred_out.label_ids

    y_true, y_pred = [], []
    for p_seq, g_seq in zip(pred_ids, labels_ids):
        t_seq, pr_seq = [], []
        for p, g in zip(p_seq, g_seq):
            if g != -100 and g < len(LABEL_LIST) and p < len(LABEL_LIST):
                t_seq.append(LABEL_LIST[g])
                pr_seq.append(LABEL_LIST[p])
        y_true.append(t_seq)
        y_pred.append(pr_seq)

    cls_report = classification_report(y_true, y_pred, digits=4)

    out_art = ART / cfg.output_subdir
    out_art.mkdir(parents=True, exist_ok=True)

    with open(out_art / 'test_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(eval_metrics, f, indent=2)

    with open(out_art / 'classification_report.txt', 'w', encoding='utf-8') as f:
        f.write(cls_report)

    pd.DataFrame([eval_metrics]).to_csv(out_art / 'test_metrics.csv', index=False)

    return {
        'config': cfg,
        'trainer': trainer_local,
        'dataset': ds,
        'tokenized': tok_ds,
        'train_output': train_output,
        'eval_metrics': eval_metrics,
        'report': cls_report,
        'artifacts_dir': out_art
    }

print('Utility training pronte.')

In [ ]:
# A1) Training SOLO italiano (lang=['it'])
# Esegui questa cella per addestrare/valutare il modello italiano.

italian_cfg = RunConfig(
    run_name='italian_only',
    langs=['it'],
    output_subdir='train_it_only'
)

# Decommenta per eseguire training completo:
# italian_run = train_and_evaluate(italian_cfg)
# print('Italian artifacts:', italian_run['artifacts_dir'])

print('Configurazione pronta:', italian_cfg)

In [ ]:
# A2) Training MULTILINGUA (italiano + inglese)
# Esegui questa cella per addestrare/valutare il modello bilingue.

multilingual_cfg = RunConfig(
    run_name='it_en_multilingual',
    langs=['it', 'en'],
    output_subdir='train_it_en'
)

# Decommenta per eseguire training completo:
# multilingual_run = train_and_evaluate(multilingual_cfg)
# print('Multilingual artifacts:', multilingual_run['artifacts_dir'])

print('Configurazione pronta:', multilingual_cfg)

## Sezione B — Benchmarking completo (modelli e dataset)

Questa sezione è indipendente dal training sopra e serve a benchmark riproducibili:

- più modelli (locali e Hugging Face)
- più benchmark NER
- output CSV/JSON per confronto nel paper

In [ ]:
# B1) Registry modelli + benchmark target
from typing import Optional

# Aggiungi/modifica modelli qui
MODEL_REGISTRY = [
    {
        'name': 'roberta_base_hf',
        'path': 'roberta-base'
    },
    {
        'name': 'italy_run_local_it',
        'path': str(ROOT / 'multinerd_checkpoints' / 'train_it_only' / 'final_model')
    },
    {
        'name': 'italy_run_local_it_en',
        'path': str(ROOT / 'multinerd_checkpoints' / 'train_it_en' / 'final_model')
    },
    {
        'name': 'xlm_roberta_base',
        'path': 'xlm-roberta-base'
    },
    {
        'name': 'bert_base_multilingual_cased',
        'path': 'bert-base-multilingual-cased'
    }
]

# Benchmark dataset specs (adatta i nomi/config in base alla tua disponibilità)
BENCHMARK_SPECS = [
    {
        'benchmark': 'multinerd_it_test',
        'loader': 'multinerd_it'
    },
    {
        'benchmark': 'multinerd_it_en_test',
        'loader': 'multinerd_it_en'
    }
]

print('Models:', len(MODEL_REGISTRY))
print('Benchmarks:', len(BENCHMARK_SPECS))

In [ ]:
# B2) Loader benchmark + funzione evaluazione unificata

def load_benchmark_dataset(loader_name: str):
    if loader_name == 'multinerd_it':
        ds = load_filtered_multinerd(['it'])
    elif loader_name == 'multinerd_it_en':
        ds = load_filtered_multinerd(['it', 'en'])
    else:
        raise ValueError(f'Loader non supportato: {loader_name}')
    return ds


def evaluate_model_on_dataset(model_path: str, dataset_for_eval: DatasetDict):
    tok_ds = tokenize_dataset(dataset_for_eval)

    model_eval = AutoModelForTokenClassification.from_pretrained(
        model_path,
        num_labels=len(LABEL_LIST),
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )

    trainer_eval = Trainer(
        model=model_eval,
        tokenizer=tokenizer,
        data_collator=data_collator
    )

    pred_out = trainer_eval.predict(tok_ds['test'])
    pred_ids = np.argmax(pred_out.predictions, axis=2)
    gold_ids = pred_out.label_ids

    y_true, y_pred = [], []
    for p_seq, g_seq in zip(pred_ids, gold_ids):
        t_seq, pr_seq = [], []
        for p, g in zip(p_seq, g_seq):
            if g != -100 and g < len(LABEL_LIST) and p < len(LABEL_LIST):
                t_seq.append(LABEL_LIST[g])
                pr_seq.append(LABEL_LIST[p])
        y_true.append(t_seq)
        y_pred.append(pr_seq)

    return {
        'precision': float(precision_score(y_true, y_pred)),
        'recall': float(recall_score(y_true, y_pred)),
        'f1': float(f1_score(y_true, y_pred)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'report': classification_report(y_true, y_pred, digits=4)
    }

print('Evaluator benchmark pronto.')

In [ ]:
# B3) Esecuzione benchmark matrix (model x benchmark) + export
benchmark_rows = []
benchmark_out_dir = ART / 'benchmark_matrix'
benchmark_out_dir.mkdir(parents=True, exist_ok=True)

for spec in BENCHMARK_SPECS:
    ds_bench = load_benchmark_dataset(spec['loader'])

    for model_spec in MODEL_REGISTRY:
        model_name = model_spec['name']
        model_path = model_spec['path']

        # Skip modelli locali non presenti
        if (model_path.startswith('/') or model_path.startswith(str(ROOT))) and (not Path(model_path).exists()):
            benchmark_rows.append({
                'benchmark': spec['benchmark'],
                'model': model_name,
                'path': model_path,
                'status': 'missing_local_model',
                'precision': None,
                'recall': None,
                'f1': None,
                'accuracy': None
            })
            continue

        try:
            res = evaluate_model_on_dataset(model_path, ds_bench)
            benchmark_rows.append({
                'benchmark': spec['benchmark'],
                'model': model_name,
                'path': model_path,
                'status': 'ok',
                'precision': res['precision'],
                'recall': res['recall'],
                'f1': res['f1'],
                'accuracy': res['accuracy']
            })

            rep_file = benchmark_out_dir / f"report__{spec['benchmark']}__{model_name}.txt"
            with open(rep_file, 'w', encoding='utf-8') as f:
                f.write(res['report'])

        except Exception as e:
            benchmark_rows.append({
                'benchmark': spec['benchmark'],
                'model': model_name,
                'path': model_path,
                'status': f'error: {e}',
                'precision': None,
                'recall': None,
                'f1': None,
                'accuracy': None
            })

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.to_csv(benchmark_out_dir / 'benchmark_matrix_results.csv', index=False)

with open(benchmark_out_dir / 'benchmark_matrix_results.json', 'w', encoding='utf-8') as f:
    json.dump(benchmark_rows, f, indent=2)

display(benchmark_df)
print('Saved:', benchmark_out_dir)

In [ ]:
# B4) Tabelle comparative pronte per paper
benchmark_out_dir = ART / 'benchmark_matrix'
benchmark_df = pd.read_csv(benchmark_out_dir / 'benchmark_matrix_results.csv')

ok_df = benchmark_df[benchmark_df['status'] == 'ok'].copy()
if len(ok_df) == 0:
    print('Nessun risultato OK disponibile.')
else:
    pivot_f1 = ok_df.pivot_table(index='model', columns='benchmark', values='f1', aggfunc='max')
    pivot_f1['avg_f1'] = pivot_f1.mean(axis=1)
    pivot_f1 = pivot_f1.sort_values('avg_f1', ascending=False)

    pivot_f1.to_csv(benchmark_out_dir / 'benchmark_f1_comparison_table.csv')
    display(pivot_f1)

    # Delta vs best per benchmark
    long_df = ok_df[['benchmark', 'model', 'f1']].copy()
    best_by_benchmark = long_df.groupby('benchmark', as_index=False)['f1'].max().rename(columns={'f1': 'best_f1'})
    merged = long_df.merge(best_by_benchmark, on='benchmark', how='left')
    merged['delta_vs_best'] = merged['f1'] - merged['best_f1']
    merged.to_csv(benchmark_out_dir / 'benchmark_delta_vs_best.csv', index=False)

    print('Saved:')
    print('-', benchmark_out_dir / 'benchmark_f1_comparison_table.csv')
    print('-', benchmark_out_dir / 'benchmark_delta_vs_best.csv')